In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc
from imblearn.over_sampling import SMOTE
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

In [2]:
df = pd.read_csv("C:\Datasets\IDS_Dataset\DoS-Wednesday-no-metadata.csv")

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Mohammad Saquib\AppData\Local\Temp\ipykernel_30140\2949570354.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv("C:\Datasets\IDS_Dataset\DoS-Wednesday-no-metadata.csv")


In [3]:
df["Label"].value_counts()

Label
Benign              391235
DoS Hulk            172846
DoS GoldenEye        10286
DoS slowloris         5385
DoS Slowhttptest      5228
Heartbleed              11
Name: count, dtype: int64

In [4]:
df.head()

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,1095,10,6,3150,3150,1575,0,315.000000,632.561650,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,15206,17,12,3452,6660,1313,0,203.058820,425.778470,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,1092,9,6,3150,3152,1575,0,350.000000,694.509700,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign


In [5]:
cols_to_drop = [
    "Fwd Packets Length Total", "Bwd Packets Length Total", "Fwd Packet Length Max", 
    "Bwd Packet Length Max", "Fwd Packet Length Min", "Bwd Packet Length Min",
    "Fwd Packet Length Std", "Bwd Packet Length Std", "Flow IAT Std", "Flow IAT Total",
    "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min", "Bwd IAT Total", "Bwd IAT Std", 
    "Bwd IAT Max", "Bwd IAT Min", "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", 
    "Bwd URG Flags", "Packet Length Variance", "PSH Flag Count", "URG Flag Count", 
    "CWE Flag Count", "ECE Flag Count", "Down/Up Ratio", "Avg Packet Size",
    "Avg Fwd Segment Size", "Avg Bwd Segment Size", "Fwd Avg Bytes/Bulk", 
    "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate", "Bwd Avg Bytes/Bulk", 
    "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate", "Subflow Bwd Bytes",
    "Init Fwd Win Bytes", "Init Bwd Win Bytes", "Fwd Act Data Packets", 
    "Fwd Seg Size Min", "Active Mean", "Active Std", "Active Max", "Active Min",
    "Idle Mean", "Idle Std", "Idle Max", "Idle Min"
]
df = df.drop(columns=cols_to_drop, axis=1, errors='ignore')

In [6]:
df.shape

(584991, 30)

In [7]:
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])
num_classes = len(le.classes_)

# Calculate Class Weights (Crucial for DoS imbalance)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df['Label']),
    y=df['Label']
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

# Scaling features
scaler = StandardScaler()
X = scaler.fit_transform(df.drop('Label', axis=1))
y = df['Label'].values

In [8]:
class DoSDetector(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(DoSDetector, self).__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.layer2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        return self.output(x)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,       
    stratify=y,          
    random_state=42
)

In [10]:
from torch.utils.data import WeightedRandomSampler, DataLoader

def train_local_model(global_weights, client_X, client_y, epochs=5):
    model = DoSDetector(input_dim=client_X.shape[1], num_classes=num_classes)
    model.load_state_dict(global_weights)
    
    class_sample_count = np.array([len(np.where(client_y == t)[0]) for t in np.unique(client_y)])
    
    weight = 1. / (class_sample_count ** 0.5)
    samples_weight = np.array([weight[t] for t in client_y])
    samples_weight = torch.from_numpy(samples_weight).double()
    
    sampler = WeightedRandomSampler(samples_weight, len(samples_weight))
    
    dataset = torch.utils.data.TensorDataset(torch.tensor(client_X, dtype=torch.float), torch.tensor(client_y, dtype=torch.long))
    
    loader = DataLoader(dataset, batch_size=64, sampler=sampler)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for epoch in range(epochs):
        for inputs, labels in loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
    return model.state_dict()

In [11]:
def federated_averaging(client_weights_list):
    # Initialize global weights with the first client's structure
    global_dict = client_weights_list[0]
    for key in global_dict.keys():
        for i in range(1, len(client_weights_list)):
            global_dict[key] += client_weights_list[i][key]
        global_dict[key] = torch.div(global_dict[key], len(client_weights_list))
    return global_dict

In [12]:
# Initialize Global Model
input_dim = X_train.shape[1] # Use training set dimensions
global_model_weights = DoSDetector(input_dim, num_classes).state_dict()

# Use 5 clients for simulation
num_clients = 5
client_data_indices = np.array_split(np.arange(len(X_train)), num_clients)

for round in range(20):
    print(f"--- Federated Round {round + 1} ---")
    local_updates = []
    
    for client_id in range(num_clients):
        # 1. Extract raw data for this client
        indices = client_data_indices[client_id]
        client_X_raw = X_train[indices]
        client_y_raw = y_train[indices]
        
        # 2. Pass the RAW arrays to the fixed training function
        # This function now handles the WeightedRandomSampler internally
        update = train_local_model(
            global_model_weights, 
            client_X_raw, 
            client_y_raw, 
            epochs=5
        )
        local_updates.append(update)
    
    # 3. Aggregate at Server
    global_model_weights = federated_averaging(local_updates)

print("Training Complete. Global model is ready for evaluation.")

--- Federated Round 1 ---
--- Federated Round 2 ---
--- Federated Round 3 ---
--- Federated Round 4 ---
--- Federated Round 5 ---
--- Federated Round 6 ---
--- Federated Round 7 ---
--- Federated Round 8 ---
--- Federated Round 9 ---
--- Federated Round 10 ---
--- Federated Round 11 ---
--- Federated Round 12 ---
--- Federated Round 13 ---
--- Federated Round 14 ---
--- Federated Round 15 ---
--- Federated Round 16 ---
--- Federated Round 17 ---
--- Federated Round 18 ---
--- Federated Round 19 ---
--- Federated Round 20 ---
Training Complete. Global model is ready for evaluation.


In [13]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_global_model(global_weights, X_test, y_test, label_encoder):
    # Initialize model and load global weights
    model = DoSDetector(input_dim=X_test.shape[1], num_classes=len(label_encoder.classes_))
    model.load_state_dict(global_weights)
    model.eval()
    
    # Convert test data to tensors
    test_X = torch.tensor(X_test, dtype=torch.float)
    test_y = torch.tensor(y_test, dtype=torch.long)
    
    with torch.no_grad():
        outputs = model(test_X)
        _, predicted = torch.max(outputs, 1)
    
    # Generate the report
    # target_names maps the numeric indices back to 'DoS Hulk', 'Benign', etc.
    report = classification_report(
        test_y.numpy(), 
        predicted.numpy(), 
        target_names=label_encoder.classes_,
        zero_division=0
    )
    
    return report

In [14]:
from sklearn.model_selection import train_test_split

# 1. Split data before the FL loop
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


# 2. After the final round, run the evaluation
print("\nFinal Global Model Performance:")
report = evaluate_global_model(global_model_weights, X_test, y_test, le)
print(report)


Final Global Model Performance:
                  precision    recall  f1-score   support

          Benign       1.00      0.97      0.99     78248
   DoS GoldenEye       0.75      1.00      0.86      2057
        DoS Hulk       0.97      0.99      0.98     34569
DoS Slowhttptest       0.86      0.99      0.92      1046
   DoS slowloris       0.69      0.98      0.81      1077
      Heartbleed       0.20      1.00      0.33         2

        accuracy                           0.98    116999
       macro avg       0.75      0.99      0.82    116999
    weighted avg       0.98      0.98      0.98    116999

